# INFO-F-422 NARX project

Statistical Foundations of Machine Learning · ULB · 2025-2026
Project notebook, team 37.

We learn a MIMO NARX model from the training pair $(\mathbf{U}^{tr}, \mathbf{Y}^{tr})$
and predict the two unseen test inputs $\mathbf{U}^{ts1}$ and $\mathbf{U}^{ts2}$.
The methodology is first developed and validated on two pilot NARXs
(eq. (5) and eq. (6) of the brief) for which the dynamics are known,
then deployed on the unknown system.

### NARX model (project spec, eq. (1))

$$
y_j(k+1) = f_j\!\Big( y_1(k-d), \ldots, y_1(k-d-n_a),\;
                       y_2(k-d), \ldots, y_2(k-d-n_a),
                       u_1(k), \ldots, u_1(k-n_b),\;
                       u_2(k), \ldots, u_2(k-n_b) \Big) + w_j(k+1)
$$

with $w_1, w_2 \sim \mathcal{N}(0, \sigma^2)$ i.i.d. The brief's
test-time initial-condition assumption is $y_1(0)=y_2(0)=u_1(0)=u_2(0)=0$
and all earlier values zero.

## 0. Setup

In [ ]:
import os, sys, pathlib, warnings

# Walk up from the notebook's directory until we find the repo root,
# put src/ on the import path, and chdir there so relative paths work.
_here = pathlib.Path.cwd().resolve()
for cand in [_here, *_here.parents]:
    if (cand / 'src').is_dir() and (cand / 'data').is_dir():
        ROOT = cand
        break
else:
    raise RuntimeError('cannot find repo root')

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

RNG_SEED = 42

### Loading the project dataset

The four arrays we are given (each shape $(1000, 2)$):

| key   | role                                           |
|-------|------------------------------------------------|
| Utr   | training inputs $u_1, u_2$                     |
| Ytr   | training outputs $y_1, y_2$                    |
| Uts1  | first test input, used for the leaderboard    |
| Uts2  | second test input, used for the project mark  |

We cast everything to `float64` since `Ytr` and `Uts2` arrive as float32.

In [ ]:
data = np.load('data/StudentdataNARX.npz')
Utr  = data['Utr'].astype(np.float64)
Ytr  = data['Ytr'].astype(np.float64)
Uts1 = data['Uts1'].astype(np.float64)
Uts2 = data['Uts2'].astype(np.float64)

for name, arr in [('Utr', Utr), ('Ytr', Ytr), ('Uts1', Uts1), ('Uts2', Uts2)]:
    print(f'{name}: shape={arr.shape}, range=[{arr.min():+.3f}, {arr.max():+.3f}]')

In [ ]:
# Quick look at the four training series.
fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax, arr, lbl in zip(axes,
                        [Utr[:, 0], Utr[:, 1], Ytr[:, 0], Ytr[:, 1]],
                        ['u1', 'u2', 'y1', 'y2']):
    ax.plot(arr, lw=0.7)
    ax.set_ylabel(lbl)
axes[-1].set_xlabel('k')
fig.suptitle('Training series')
plt.tight_layout()
plt.show()

## 1. Pilot simulators (Task 1)

The two pilots in §3 of the brief have closed-form dynamics, so we can
use them as known ground-truth systems to validate every step of the
pipeline (lag selection, fitting, multi-step prediction) before
betting anything on the unknown training data.

### 1.1 NARX1: single input, two coupled outputs

$$
\begin{aligned}
y_1(k{+}1) &= 0.5\, y_2(k{-}1) + \sin\bigl(y_2(k)\bigr) + 0.3\, u(k{-}1) + w_1(k{+}1)\\
y_2(k{+}1) &= 0.5\, y_1(k{-}1) + \sin\bigl(y_1(k)\bigr) + 0.2\, u(k)   + w_2(k{+}1)
\end{aligned}
$$

Reading the equation directly: $y_1(k{+}1)$ and $y_2(k{+}1)$ both need
the previous output at lags $0$ and $1$, and $u$ at lags $0$ and $1$.
True structural parameters are therefore $d{=}0$, $n_a{=}1$, $n_b{=}1$.

In [ ]:
from narx.simulators import (simulate_narx1, simulate_narx2,
                               NARX1_TRUE_PARAMS, NARX2_TRUE_PARAMS)

# Pilot lengths matching the real data (1000 train, 1000 test) so the
# methodology validation runs on a regime comparable to what we'll
# encounter at deployment.
N_tr, N_ts = 1000, 1000
U1tr, Y1tr, U1ts, Y1ts = simulate_narx1(N_tr, N_ts, sigma=0.05, seed=RNG_SEED)
U2tr, Y2tr, U2ts, Y2ts = simulate_narx2(N_tr, N_ts, sigma=0.01, seed=RNG_SEED)

print(f'NARX1: train U{U1tr.shape} Y{Y1tr.shape}  test U{U1ts.shape} Y{Y1ts.shape}')
print(f'NARX2: train U{U2tr.shape} Y{Y2tr.shape}  test U{U2ts.shape} Y{Y2ts.shape}')

In [ ]:
# Pilot training signals at a glance.
def stylize(ax):
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

step = 2
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Pilot NARX training signals')

k1 = np.arange(len(U1tr))
axes[0, 0].plot(k1[::step], U1tr[::step, 0], lw=1.0, color='black')
axes[0, 0].set_title('NARX1: u(k)'); axes[0, 0].set_xlabel('k'); stylize(axes[0, 0])
axes[0, 1].plot(k1[::step], Y1tr[::step, 0], lw=1.1, color='royalblue')
axes[0, 1].set_title('NARX1: y1(k)'); axes[0, 1].set_xlabel('k'); stylize(axes[0, 1])
axes[0, 2].plot(k1[::step], Y1tr[::step, 1], lw=1.1, color='seagreen')
axes[0, 2].set_title('NARX1: y2(k)'); axes[0, 2].set_xlabel('k'); stylize(axes[0, 2])

k2 = np.arange(len(U2tr))
axes[1, 0].plot(k2[::step], U2tr[::step, 0], lw=1.0, color='dodgerblue', label='u1')
axes[1, 0].plot(k2[::step], U2tr[::step, 1], lw=1.0, color='crimson', alpha=0.8, label='u2')
axes[1, 0].set_title('NARX2: inputs'); axes[1, 0].legend(); stylize(axes[1, 0])
axes[1, 1].plot(k2[::step], Y2tr[::step, 0], lw=1.1, color='darkorange')
axes[1, 1].set_title('NARX2: y1(k)'); axes[1, 1].set_xlabel('k'); stylize(axes[1, 1])
axes[1, 2].plot(k2[::step], Y2tr[::step, 1], lw=1.1, color='saddlebrown')
axes[1, 2].set_title('NARX2: y2(k)'); axes[1, 2].set_xlabel('k'); stylize(axes[1, 2])

plt.tight_layout(); plt.show()

### 1.2 NARX2: two inputs, rational nonlinearity

$$
\begin{aligned}
y_1(k{+}1) &= \frac{y_1(k)\,y_1(k{-}1)\,y_1(k{-}2)\,\bigl(y_1(k{-}2){-}1\bigr)\,u_2(k{-}1) + u_2(k)}{1 + y_2(k{-}1)^2 + y_2(k{-}2)^2} + w_1(k{+}1)\\[6pt]
y_2(k{+}1) &= \frac{y_2(k)\,y_2(k{-}1)\,y_2(k{-}2)\,\bigl(y_2(k{-}2){-}1\bigr)\,u_1(k{-}1) + u_1(k)}{1 + y_1(k{-}1)^2 + y_1(k{-}2)^2} + w_2(k{+}1)
\end{aligned}
$$

Three output lags appear (deepest is $y(k{-}2)$) and $u$ at lags $0,1$.
True structural parameters: $d{=}0$, $n_a{=}2$, $n_b{=}1$.
We squeeze the input range to $[-0.5, 0.5]$ so the rational map stays
well-behaved near the origin.